# GéoMarketing IDF — J7 : Profil détaillé de la clientèle potentielle

## 07b - Couples, ménages et familles

In [1]:
#Importation des librairies
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from zipfile import ZipFile
import re
import shutil
import unicodedata

import numpy as np
import pandas as pd
import openpyxl

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)

print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("Openpyxl :", openpyxl.__version__)
print("Importations réussies ✅")

Pandas : 2.2.2
NumPy : 1.26.4
Openpyxl : 3.1.5
Importations réussies ✅


In [2]:
#Dossiers
RACINE = Path(
    r"C:\Users\almou\OneDrive\GeoMarketing_IDF"
)

DOSSIER_RAW = (
    RACINE
    / "data"
    / "raw"
    / "insee"
    / "rp2023"
)

DOSSIER_INTERIM = (
    RACINE
    / "data"
    / "interim"
)

DOSSIER_PROCESSED = (
    RACINE
    / "data"
    / "processed"
)

for dossier in [
    DOSSIER_RAW,
    DOSSIER_INTERIM,
    DOSSIER_PROCESSED,
]:
    dossier.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Racine :", RACINE)
print("Raw :", DOSSIER_RAW)
print("Interim :", DOSSIER_INTERIM)
print("Processed :", DOSSIER_PROCESSED)

assert RACINE.exists(), "Le dossier du projet n'existe pas."


Racine : C:\Users\almou\OneDrive\GeoMarketing_IDF
Raw : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\rp2023
Interim : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim
Processed : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed


In [3]:
def normaliser_nom_colonne(nom):
    nom = str(nom).strip().upper()

    nom = unicodedata.normalize(
        "NFKD",
        nom,
    )

    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )

    nom = re.sub(
        r"[^A-Z0-9]+",
        "_",
        nom,
    )

    return nom.strip("_")


def normaliser_code_commune(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
        .str.upper()
        .str.zfill(5)
    )


def pourcentage(numerateur, denominateur):
    numerateur = pd.to_numeric(
        numerateur,
        errors="coerce",
    )

    denominateur = pd.to_numeric(
        denominateur,
        errors="coerce",
    )

    return (
        numerateur
        .div(
            denominateur.where(
                denominateur.ne(0)
            )
        )
        .mul(100)
    )


def verifier_classeur_xlsx(fichier):
    if not fichier.exists():
        raise FileNotFoundError(
            f"Classeur introuvable : {fichier}"
        )

    with open(fichier, "rb") as flux:
        signature = flux.read(4)

    if signature != b"PK\x03\x04":
        raise ValueError(
            f"{fichier.name} n'est pas un véritable fichier XLSX."
        )

    with ZipFile(fichier) as archive:
        noms = set(archive.namelist())

        if "xl/workbook.xml" not in noms:
            raise ValueError(
                f"{fichier.name} ne contient pas de classeur Excel valide."
            )

    print(
        f"Classeur valide : {fichier.name} "
        f"({fichier.stat().st_size / 1_000_000:.2f} Mo)"
    )


def enregistrer_csv(table, fichier):
    table.to_csv(
        fichier,
        index=False,
        sep=",",
        encoding="utf-8-sig",
    )

    print(
        "Fichier CSV créé :",
        fichier,
    )

In [4]:
noms_profils_j6 = [
    "profil_communes_idf_j6.xlsx",
    "profil_communes_idf_j6.csv",
    "profil_communes_idf.csv",
]

FICHIER_PROFIL_J6 = None

for nom in noms_profils_j6:
    candidat = DOSSIER_PROCESSED / nom

    if candidat.exists():
        FICHIER_PROFIL_J6 = candidat
        break

if FICHIER_PROFIL_J6 is None:
    raise FileNotFoundError(
        "Le profil J6 est introuvable dans data/processed."
    )

if FICHIER_PROFIL_J6.suffix.lower() == ".xlsx":
    profil_j6 = pd.read_excel(
        FICHIER_PROFIL_J6,
        sheet_name=0,
        engine="openpyxl",
    )

else:
    profil_j6 = pd.read_csv(
        FICHIER_PROFIL_J6,
        sep=None,
        engine="python",
        encoding="utf-8-sig",
    )

profil_j6.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil_j6.columns
]

if "CODGEO" not in profil_j6.columns:
    candidats_code = [
        "DEPCOM",
        "CODE_COMMUNE",
        "COM",
    ]

    colonne_code = next(
        (
            colonne
            for colonne in candidats_code
            if colonne in profil_j6.columns
        ),
        None,
    )

    if colonne_code is None:
        raise ValueError(
            "Aucune colonne de code communal dans le profil J6."
        )

    profil_j6 = profil_j6.rename(
        columns={
            colonne_code: "CODGEO"
        }
    )

profil_j6["CODGEO"] = normaliser_code_commune(
    profil_j6["CODGEO"]
)

assert profil_j6["CODGEO"].notna().all()
assert profil_j6["CODGEO"].is_unique
assert profil_j6["CODGEO"].str.fullmatch(
    r"\d{5}"
).all()

codes_profil = set(
    profil_j6["CODGEO"]
)

print("Profil J6 :", FICHIER_PROFIL_J6)
print("Nombre de communes :", len(profil_j6))
print("Profil J6 chargé ✅")

Profil J6 : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j6.csv
Nombre de communes : 1266
Profil J6 chargé ✅


In [5]:
FICHIER_FAMILLES = (
    DOSSIER_RAW
    / "base_cc_coupl_fam-men_2023.xlsx"
)

verifier_classeur_xlsx(
    FICHIER_FAMILLES
)

classeur_familles = pd.ExcelFile(
    FICHIER_FAMILLES,
    engine="openpyxl",
)

print(
    "Onglets :",
    classeur_familles.sheet_names,
)

assert "COM_2023" in classeur_familles.sheet_names

Classeur valide : base_cc_coupl_fam-men_2023.xlsx (52.00 Mo)
Onglets : ['Métadonnées', 'COM_2023', 'ARM_2023', 'COM_2017', 'ARM_2017', 'COM_2012', 'ARM_2012', 'Documentation']


In [6]:
#Définir les colonnes nécessaires

colonnes_familles = {
    "Code géographique": "CODGEO",
    "Libellé géographique": "LIBGEO",

    "Ménages (compl)": "NB_MENAGES",
    "Ménages 1 personne (compl)": "NB_MENAGES_1_PERSONNE",
    "Ménages Hommes seuls (compl)": "NB_MENAGES_HOMMES_SEULS",
    "Ménages Femmes seules (compl)": "NB_MENAGES_FEMMES_SEULES",
    "Ménages Autres sans famille (compl)": "NB_MENAGES_AUTRES_SANS_FAMILLE",
    "Ménages avec famille(s) (compl)": "NB_MENAGES_AVEC_FAMILLES",

    "Mén fam princ Couple sans enfant (compl)": "NB_MENAGES_COUPLE_SANS_ENFANT",
    "Mén fam princ Couple avec enfant(s) (compl)": "NB_MENAGES_COUPLE_AVEC_ENFANTS",
    "Mén fam princ Famille mono (compl)": "NB_MENAGES_FAMILLE_MONOPARENTALE",

    "Pop Ménages (compl)": "POP_MENAGES",
    "Pop 15 ans ou plus (princ)": "POP_15_PLUS",

    "Pop 15 ans ou plus mariée (princ)": "POP_15P_MARIEE",
    "Pop 15 ans ou plus pacsée (princ)": "POP_15P_PACSEE",
    "Pop 15 ans ou plus en concubinage ou union libre (princ)": "POP_15P_CONCUBINAGE",
    "Pop 15 ans ou plus veuves ou veufs (princ)": "POP_15P_VEUVE_VEUF",
    "Pop 15 ans ou plus divorcée (princ)": "POP_15P_DIVORCEE",
    "Pop 15 ans ou plus célibataire (princ)": "POP_15P_CELIBATAIRE",

    "Ménages Pers Réf Agri. expl. (compl)": "NB_MENAGES_REF_AGRICULTEUR",
    "Ménages Pers Réf Art. Comm. Chefs entr. (compl)": "NB_MENAGES_REF_ARTISAN_COMMERCANT_CHEF",
    "Ménages Pers Réf Cadres Prof int sup (compl)": "NB_MENAGES_REF_CADRE",
    "Ménages Pers Réf Prof intermédiaire (compl)": "NB_MENAGES_REF_PROF_INTERMEDIAIRE",
    "Ménages Pers Réf Employé (compl)": "NB_MENAGES_REF_EMPLOYE",
    "Ménages Pers Réf Ouvrier (compl)": "NB_MENAGES_REF_OUVRIER",
    "Ménages Pers Réf Retraité (compl)": "NB_MENAGES_REF_RETRAITE",
    "Ménages Pers Réf Autre (compl)": "NB_MENAGES_REF_AUTRE",

    "Familles (compl)": "NB_FAMILLES",
    "Fam Couple avec enfant(s) (compl)": "NB_FAMILLES_COUPLE_AVEC_ENFANTS",
    "Fam Monoparentales (compl)": "NB_FAMILLES_MONOPARENTALES",
    "Fam Mono Hommes avec enfant(s) (compl)": "NB_FAMILLES_MONO_HOMMES",
    "Fam Mono Femmes avec enfant(s) (compl)": "NB_FAMILLES_MONO_FEMMES",
    "Fam Couple sans enfant (compl)": "NB_FAMILLES_COUPLE_SANS_ENFANT",

    "Fam 0 enfant moins 25 ans (compl)": "NB_FAMILLES_0_ENFANT_MOINS25",
    "Fam 1 enfant moins 25 ans (compl)": "NB_FAMILLES_1_ENFANT_MOINS25",
    "Fam 2 enfants moins 25 ans (compl)": "NB_FAMILLES_2_ENFANTS_MOINS25",
    "Fam 3 enfants moins 25 ans (compl)": "NB_FAMILLES_3_ENFANTS_MOINS25",
    "Fam 4 enfants ou plus moins 25 ans (compl)": "NB_FAMILLES_4PLUS_ENFANTS_MOINS25",
}

tranches_relation = {
    "15_24": "15-24 ans",
    "25_39": "25-39 ans",
    "40_54": "40-54 ans",
    "55_64": "55-64 ans",
    "65_79": "65-79 ans",
    "80_PLUS": "80 ans ou plus",
}

for code_tranche, libelle_tranche in tranches_relation.items():
    colonnes_familles[
        f"Pop mén {libelle_tranche} (princ)"
    ] = f"POP_MENAGES_{code_tranche}"

    colonnes_familles[
        f"Pop {libelle_tranche} vivant seule (princ)"
    ] = f"POP_VIVANT_SEULE_{code_tranche}"

    colonnes_familles[
        f"Pop {libelle_tranche} vivant en couple (princ)"
    ] = f"POP_VIVANT_EN_COUPLE_{code_tranche}"

print(
    "Nombre de colonnes demandées :",
    len(colonnes_familles),
)

Nombre de colonnes demandées : 56


In [7]:
#Vérifier puis lire les colonnes
entete_familles = pd.read_excel(
    FICHIER_FAMILLES,
    sheet_name="COM_2023",
    nrows=0,
    engine="openpyxl",
)

colonnes_absentes = (
    set(colonnes_familles)
    - set(entete_familles.columns)
)

if colonnes_absentes:
    raise ValueError(
        "Colonnes absentes du classeur familles : "
        f"{sorted(colonnes_absentes)}"
    )

familles_source = pd.read_excel(
    FICHIER_FAMILLES,
    sheet_name="COM_2023",
    usecols=list(colonnes_familles),
    dtype={
        "Code géographique": "string",
    },
    engine="openpyxl",
)

familles_source = familles_source.rename(
    columns=colonnes_familles
)

familles_source["CODGEO"] = (
    normaliser_code_commune(
        familles_source["CODGEO"]
    )
)

print(
    "Dimensions nationales :",
    familles_source.shape,
)

display(
    familles_source.head()
)

Dimensions nationales : (34858, 56)


,CODGEO,LIBGEO,NB_MENAGES,NB_MENAGES_1_PERSONNE,NB_MENAGES_HOMMES_SEULS,NB_MENAGES_FEMMES_SEULES,NB_MENAGES_AUTRES_SANS_FAMILLE,NB_MENAGES_AVEC_FAMILLES,NB_MENAGES_COUPLE_SANS_ENFANT,NB_MENAGES_COUPLE_AVEC_ENFANTS,NB_MENAGES_FAMILLE_MONOPARENTALE,POP_MENAGES,POP_15_PLUS,POP_MENAGES_15_24,POP_MENAGES_25_39,POP_MENAGES_40_54,POP_MENAGES_55_64,POP_MENAGES_65_79,POP_MENAGES_80_PLUS,POP_VIVANT_SEULE_15_24,POP_VIVANT_SEULE_25_39,POP_VIVANT_SEULE_40_54,POP_VIVANT_SEULE_55_64,POP_VIVANT_SEULE_65_79,POP_VIVANT_SEULE_80_PLUS,POP_VIVANT_EN_COUPLE_15_24,POP_VIVANT_EN_COUPLE_25_39,POP_VIVANT_EN_COUPLE_40_54,POP_VIVANT_EN_COUPLE_55_64,POP_VIVANT_EN_COUPLE_65_79,POP_VIVANT_EN_COUPLE_80_PLUS,POP_15P_MARIEE,POP_15P_PACSEE,POP_15P_CONCUBINAGE,POP_15P_VEUVE_VEUF,POP_15P_DIVORCEE,POP_15P_CELIBATAIRE,NB_MENAGES_REF_AGRICULTEUR,NB_MENAGES_REF_ARTISAN_COMMERCANT_CHEF,NB_MENAGES_REF_CADRE,NB_MENAGES_REF_PROF_INTERMEDIAIRE,NB_MENAGES_REF_EMPLOYE,NB_MENAGES_REF_OUVRIER,NB_MENAGES_REF_RETRAITE,NB_MENAGES_REF_AUTRE,NB_FAMILLES,NB_FAMILLES_COUPLE_AVEC_ENFANTS,NB_FAMILLES_MONOPARENTALES,NB_FAMILLES_MONO_HOMMES,NB_FAMILLES_MONO_FEMMES,NB_FAMILLES_COUPLE_SANS_ENFANT,NB_FAMILLES_0_ENFANT_MOINS25,NB_FAMILLES_1_ENFANT_MOINS25,NB_FAMILLES_2_ENFANTS_MOINS25,NB_FAMILLES_3_ENFANTS_MOINS25,NB_FAMILLES_4PLUS_ENFANTS_MOINS25
0,01001,L'Abergement-Clémenciat,355.23341,74.76105,44.85663,29.90442,9.89751,270.57484,118.77014,127.09598,24.70872,860.00000,699.19487,65.80238,150.78022,177.08301,131.92646,129.14552,44.45728,2.03059,9.13766,12.18354,9.13766,23.35179,16.24473,3.98256,125.63411,154.83988,118.78501,97.77876,23.20792,358.15106,54.91889,115.19085,41.51231,22.18604,107.23572,4.91369,14.81120,48.96408,59.24557,44.25929,63.98666,114.10415,4.94876,270.57484,127.09598,24.70872,9.86245,14.84627,118.77014,133.58135,68.82700,48.82358,14.53292,4.80998
1,01002,L'Abergement-de-Varey,118.78183,40.25025,30.18769,10.06256,0.00000,78.53158,34.77034,38.85732,4.90392,270.00000,214.19188,24.35625,40.39771,73.08923,26.06338,32.19707,18.08824,1.01423,6.08540,10.14233,8.11387,12.17080,5.07117,0.00000,29.40539,58.99300,16.02102,19.02496,13.01708,102.12448,23.67536,20.81685,7.08671,16.02489,44.46358,9.68290,4.84145,14.83912,14.96648,4.96719,19.49233,49.99236,0.00000,78.53158,38.85732,4.90392,0.00000,4.90392,34.77034,34.77034,14.71175,24.26971,4.77978,0.00000
2,01004,Ambérieu-en-Bugey,7204.31721,2952.78523,1296.77620,1656.00903,103.20950,4148.32248,1697.74596,1799.79838,650.77814,15430.01402,12828.66776,1925.31250,3229.68022,2791.79000,1812.06457,1961.14749,686.02003,316.60074,570.41354,548.65242,548.82291,711.77035,371.38737,325.65909,2326.71141,1854.93157,1127.60628,1236.20136,302.69416,5193.79570,836.36106,1310.59140,881.41653,1064.71862,3541.78446,16.37946,269.26682,832.10571,1349.81280,1232.71767,1288.15802,1892.38198,323.49475,4178.70973,1804.76704,660.58499,126.40875,534.17624,1713.35769,1910.72382,923.35862,855.42341,384.79596,104.40790
3,01005,Ambérieux-en-Dombes,826.77126,227.34457,79.87782,147.46675,28.63687,570.78982,252.00443,279.08254,39.70285,1906.00000,1553.97385,167.04194,384.96861,398.17898,272.08890,243.31730,88.37813,7.37513,35.64647,51.62593,50.39674,44.25080,31.95891,31.02963,300.22810,310.12428,203.72940,191.79973,46.82035,755.83436,120.17765,220.93526,74.16183,71.99948,310.86528,0.00000,55.49030,103.61198,153.77235,108.33841,153.32888,240.35752,11.87182,570.78982,279.08254,39.70285,16.79336,22.90949,252.00443,289.76346,130.07044,114.16312,32.46919,4.32361
4,01006,Ambléon,57.50000,15.68182,15.68182,0.00000,0.00000,41.81818,20.90909,10.45455,10.45455,115.00000,104.91228,8.07018,22.19298,18.15789,24.21053,26.22807,6.05263,0.00000,2.01754,5.04386,6.05263,6.05263,0.00000,1.00877,13.11404,13.11404,17.14912,20.17544,5.04386,51.44737,9.07895,11.09649,3.02632,10.08772,20.17544,0.00000,0.00000,0.00000,5.22727,10.45455,26.13636,10.45455,5.22727,41.81818,10.45455,10.45455,0.00000,10.45455,20.90909,20.90909,20.90909,0.00000,0.00000,0.00000


In [8]:
#Convertir les effectifs pour filtrer uniquement les communes d'Ile-de-France
colonnes_numeriques_familles = [
    colonne
    for colonne in familles_source.columns
    if colonne not in [
        "CODGEO",
        "LIBGEO",
    ]
]

familles_source[
    colonnes_numeriques_familles
] = familles_source[
    colonnes_numeriques_familles
].apply(
    pd.to_numeric,
    errors="coerce",
)

codes_familles = set(
    familles_source["CODGEO"]
)

codes_absents_familles = sorted(
    codes_profil - codes_familles
)

if codes_absents_familles:
    raise ValueError(
        "Codes du profil absents du classeur familles : "
        f"{codes_absents_familles}"
    )

familles_idf_source = familles_source[
    familles_source["CODGEO"].isin(
        codes_profil
    )
].copy()

familles_idf_source = (
    familles_idf_source
    .sort_values("CODGEO")
    .reset_index(drop=True)
)

assert len(familles_idf_source) == len(profil_j6)
assert familles_idf_source["CODGEO"].is_unique

print(
    "Communes IDF :",
    len(familles_idf_source),
)

Communes IDF : 1266


In [9]:
#Calculer les indicateurs liés au ménage et à la famille
indicateurs_7b = familles_idf_source.copy()

indicateurs_7b["TAILLE_MOYENNE_MENAGE"] = (
    indicateurs_7b["POP_MENAGES"]
    .div(
        indicateurs_7b[
            "NB_MENAGES"
        ].where(
            indicateurs_7b[
                "NB_MENAGES"
            ].ne(0)
        )
    )
)

indicateurs_7b[
    "PART_MENAGES_1_PERSONNE_PCT"
] = pourcentage(
    indicateurs_7b[
        "NB_MENAGES_1_PERSONNE"
    ],
    indicateurs_7b["NB_MENAGES"],
)

indicateurs_7b[
    "PART_MENAGES_AVEC_FAMILLES_PCT"
] = pourcentage(
    indicateurs_7b[
        "NB_MENAGES_AVEC_FAMILLES"
    ],
    indicateurs_7b["NB_MENAGES"],
)

indicateurs_7b[
    "PART_MENAGES_COUPLE_SANS_ENFANT_PCT"
] = pourcentage(
    indicateurs_7b[
        "NB_MENAGES_COUPLE_SANS_ENFANT"
    ],
    indicateurs_7b["NB_MENAGES"],
)

indicateurs_7b[
    "PART_MENAGES_COUPLE_AVEC_ENFANTS_PCT"
] = pourcentage(
    indicateurs_7b[
        "NB_MENAGES_COUPLE_AVEC_ENFANTS"
    ],
    indicateurs_7b["NB_MENAGES"],
)

indicateurs_7b[
    "PART_MENAGES_FAMILLE_MONOPARENTALE_PCT"
] = pourcentage(
    indicateurs_7b[
        "NB_MENAGES_FAMILLE_MONOPARENTALE"
    ],
    indicateurs_7b["NB_MENAGES"],
)

indicateurs_7b[
    "NB_FAMILLES_AVEC_ENFANTS_MOINS25"
] = (
    indicateurs_7b[
        "NB_FAMILLES_1_ENFANT_MOINS25"
    ]
    + indicateurs_7b[
        "NB_FAMILLES_2_ENFANTS_MOINS25"
    ]
    + indicateurs_7b[
        "NB_FAMILLES_3_ENFANTS_MOINS25"
    ]
    + indicateurs_7b[
        "NB_FAMILLES_4PLUS_ENFANTS_MOINS25"
    ]
)

indicateurs_7b[
    "PART_FAMILLES_AVEC_ENFANTS_MOINS25_PCT"
] = pourcentage(
    indicateurs_7b[
        "NB_FAMILLES_AVEC_ENFANTS_MOINS25"
    ],
    indicateurs_7b["NB_FAMILLES"],
)

indicateurs_7b[
    "PART_FAMILLES_MONOPARENTALES_PCT"
] = pourcentage(
    indicateurs_7b[
        "NB_FAMILLES_MONOPARENTALES"
    ],
    indicateurs_7b["NB_FAMILLES"],
)

indicateurs_7b[
    "PART_FAMILLES_3PLUS_ENFANTS_MOINS25_PCT"
] = pourcentage(
    indicateurs_7b[
        "NB_FAMILLES_3_ENFANTS_MOINS25"
    ]
    + indicateurs_7b[
        "NB_FAMILLES_4PLUS_ENFANTS_MOINS25"
    ],
    indicateurs_7b["NB_FAMILLES"],
)

In [10]:
# Part des personnes seules
colonnes_seules = []
colonnes_couples = []

for code_tranche in tranches_relation:
    colonne_population = (
        f"POP_MENAGES_{code_tranche}"
    )

    colonne_seule = (
        f"POP_VIVANT_SEULE_{code_tranche}"
    )

    colonne_couple = (
        f"POP_VIVANT_EN_COUPLE_{code_tranche}"
    )

    colonnes_seules.append(
        colonne_seule
    )

    colonnes_couples.append(
        colonne_couple
    )

    indicateurs_7b[
        f"PART_VIVANT_SEULE_{code_tranche}_PCT"
    ] = pourcentage(
        indicateurs_7b[colonne_seule],
        indicateurs_7b[colonne_population],
    )

    indicateurs_7b[
        f"PART_VIVANT_EN_COUPLE_{code_tranche}_PCT"
    ] = pourcentage(
        indicateurs_7b[colonne_couple],
        indicateurs_7b[colonne_population],
    )

indicateurs_7b[
    "POP_VIVANT_SEULE_15_PLUS"
] = indicateurs_7b[
    colonnes_seules
].sum(axis=1)

indicateurs_7b[
    "POP_VIVANT_EN_COUPLE_15_PLUS"
] = indicateurs_7b[
    colonnes_couples
].sum(axis=1)

indicateurs_7b[
    "PART_VIVANT_SEULE_15_PLUS_PCT"
] = pourcentage(
    indicateurs_7b[
        "POP_VIVANT_SEULE_15_PLUS"
    ],
    indicateurs_7b["POP_15_PLUS"],
)

indicateurs_7b[
    "PART_VIVANT_EN_COUPLE_15_PLUS_PCT"
] = pourcentage(
    indicateurs_7b[
        "POP_VIVANT_EN_COUPLE_15_PLUS"
    ],
    indicateurs_7b["POP_15_PLUS"],
)

In [11]:
# Part par statut conjugal et CSP

situations_familiales = [
    "MARIEE",
    "PACSEE",
    "CONCUBINAGE",
    "VEUVE_VEUF",
    "DIVORCEE",
    "CELIBATAIRE",
]

for situation in situations_familiales:
    indicateurs_7b[
        f"PART_15P_{situation}_PCT"
    ] = pourcentage(
        indicateurs_7b[
            f"POP_15P_{situation}"
        ],
        indicateurs_7b["POP_15_PLUS"],
    )


categories_menages = [
    "AGRICULTEUR",
    "ARTISAN_COMMERCANT_CHEF",
    "CADRE",
    "PROF_INTERMEDIAIRE",
    "EMPLOYE",
    "OUVRIER",
    "RETRAITE",
    "AUTRE",
]

for categorie in categories_menages:
    indicateurs_7b[
        f"PART_MENAGES_REF_{categorie}_PCT"
    ] = pourcentage(
        indicateurs_7b[
            f"NB_MENAGES_REF_{categorie}"
        ],
        indicateurs_7b["NB_MENAGES"],
    )

In [12]:
# Vérifier s'il y a des écarts entre les données brutes et les données agrégées

def ecart_composantes(
    table,
    total,
    composantes,
):
    return (
        table[total]
        - table[composantes].sum(axis=1)
    ).abs()


ecart_types_menages = ecart_composantes(
    indicateurs_7b,
    "NB_MENAGES",
    [
        "NB_MENAGES_1_PERSONNE",
        "NB_MENAGES_AUTRES_SANS_FAMILLE",
        "NB_MENAGES_AVEC_FAMILLES",
    ],
)

ecart_types_familles = ecart_composantes(
    indicateurs_7b,
    "NB_FAMILLES",
    [
        "NB_FAMILLES_COUPLE_AVEC_ENFANTS",
        "NB_FAMILLES_MONOPARENTALES",
        "NB_FAMILLES_COUPLE_SANS_ENFANT",
    ],
)

ecart_nombre_enfants = ecart_composantes(
    indicateurs_7b,
    "NB_FAMILLES",
    [
        "NB_FAMILLES_0_ENFANT_MOINS25",
        "NB_FAMILLES_1_ENFANT_MOINS25",
        "NB_FAMILLES_2_ENFANTS_MOINS25",
        "NB_FAMILLES_3_ENFANTS_MOINS25",
        "NB_FAMILLES_4PLUS_ENFANTS_MOINS25",
    ],
)

controle_7b = pd.DataFrame(
    {
        "CONTROLE": [
            "Nombre de communes",
            "Écart maximal types de ménages",
            "Écart maximal types de familles",
            "Écart maximal nombre d'enfants",
            "Présence de Paris 75056",
            "Présence de Saint-Denis 93066",
        ],
        "VALEUR": [
            len(indicateurs_7b),
            ecart_types_menages.max(),
            ecart_types_familles.max(),
            ecart_nombre_enfants.max(),
            "75056" in indicateurs_7b["CODGEO"].values,
            "93066" in indicateurs_7b["CODGEO"].values,
        ],
    }
)

assert indicateurs_7b["CODGEO"].is_unique
assert len(indicateurs_7b) == len(profil_j6)
assert ecart_types_menages.max() < 0.1
assert ecart_types_familles.max() < 0.1
assert ecart_nombre_enfants.max() < 0.1

display(controle_7b)

print("Contrôles J7b validés ✅")

,CONTROLE,VALEUR
0,Nombre de communes,1266
1,Écart maximal types de ménages,0.00001
2,Écart maximal types de familles,0.00001
3,Écart maximal nombre d'enfants,0.00002
4,Présence de Paris 75056,True
5,Présence de Saint-Denis 93066,True


Contrôles J7b validés ✅


In [13]:
FICHIER_SORTIE_7B = (
    DOSSIER_INTERIM
    / "j7b_menages_familles_idf_2023.csv"
)

FICHIER_CONTROLE_7B = (
    DOSSIER_INTERIM
    / "j7b_controle_menages_familles.csv"
)

enregistrer_csv(
    indicateurs_7b,
    FICHIER_SORTIE_7B,
)

enregistrer_csv(
    controle_7b,
    FICHIER_CONTROLE_7B,
)

print("J7b terminé ✅")

Fichier CSV créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j7b_menages_familles_idf_2023.csv
Fichier CSV créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j7b_controle_menages_familles.csv
J7b terminé ✅
